# Phase 1-5: 채널 간 연결성 분석

**목표**: 뇌 네트워크의 class별 차이 확인 → EDCC Stage 3 GCN 설계 근거

**분석 내용**:
1. 채널 간 상관관계 행렬 (19x19) — class별
2. 뇌 영역 간 평균 연결 강도 (5x5) — class별
3. EO vs EC에서의 연결성 변화 (class별)
4. Graph metrics: clustering coefficient, characteristic path length
5. Class별 연결성 차이의 통계적 유의성

In [ ]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sys.path.insert(0, os.path.abspath('..'))

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

DATASET_PATH = '../local/dataset/caueeg-dataset'
SAMPLING_RATE = 200

CHANNEL_NAMES = ['Fp1', 'F3', 'C3', 'P3', 'O1', 'Fp2', 'F4', 'C4', 'P4', 'O2',
                 'F7', 'T3', 'T5', 'F8', 'T4', 'T6', 'FZ', 'CZ', 'PZ']

BRAIN_REGIONS = {
    'Frontal':  [0, 5, 1, 6, 10, 13, 16],
    'Central':  [2, 7, 17],
    'Parietal': [3, 8, 18],
    'Temporal': [11, 14, 12, 15],
    'Occipital':[4, 9],
}

# 채널→영역 매핑
ch_to_region = {}
for region, indices in BRAIN_REGIONS.items():
    for idx in indices:
        ch_to_region[idx] = region

In [ ]:
from datasets.caueeg_script import load_caueeg_task_datasets

task_config, train_ds, val_ds, test_ds = load_caueeg_task_datasets(
    DATASET_PATH, task='dementia', load_event=True, file_format='edf'
)
class_names = task_config['class_label_to_name']
colors = {'Normal': '#2ecc71', 'MCI': '#f39c12', 'Dementia': '#e74c3c'}

## 1. Class별 채널 간 상관관계 행렬 (19x19)

녹음의 중간 60초 구간에서 채널 간 상관관계를 계산합니다.

In [ ]:
# Class별 평균 상관관계 행렬 계산
print("Computing correlation matrices...")

class_corr_matrices = {cls: [] for cls in ['Normal', 'MCI', 'Dementia']}

for i in range(len(train_ds)):
    sample = train_ds[i]
    eeg = sample['signal'][:19]
    cls = class_names[sample['class_label']]
    
    # 녹음 중간 60초 구간
    mid = eeg.shape[1] // 2
    start = max(0, mid - 6000)
    end = min(eeg.shape[1], mid + 6000)
    eeg_seg = eeg[:, start:end]
    
    # Pearson 상관관계 행렬
    corr_matrix = np.corrcoef(eeg_seg)
    if not np.any(np.isnan(corr_matrix)):
        class_corr_matrices[cls].append(corr_matrix)
    
    if (i + 1) % 200 == 0:
        print(f"  {i+1}/{len(train_ds)} processed")

# Class별 평균 행렬
avg_corr = {}
for cls in ['Normal', 'MCI', 'Dementia']:
    avg_corr[cls] = np.mean(class_corr_matrices[cls], axis=0)
    print(f"{cls}: {len(class_corr_matrices[cls])} matrices")

# 시각화
fig, axes = plt.subplots(1, 4, figsize=(22, 5))

for ax_idx, cls in enumerate(['Normal', 'MCI', 'Dementia']):
    sns.heatmap(avg_corr[cls], annot=False, cmap='RdBu_r', vmin=-0.5, vmax=1.0,
                xticklabels=CHANNEL_NAMES, yticklabels=CHANNEL_NAMES,
                ax=axes[ax_idx], square=True)
    axes[ax_idx].set_title(f'{cls}', fontweight='bold')
    axes[ax_idx].tick_params(labelsize=7)

# Difference: Normal - Dementia
diff = avg_corr['Normal'] - avg_corr['Dementia']
sns.heatmap(diff, annot=False, cmap='RdBu_r', vmin=-0.3, vmax=0.3,
            xticklabels=CHANNEL_NAMES, yticklabels=CHANNEL_NAMES,
            ax=axes[3], square=True)
axes[3].set_title('Normal - Dementia', fontweight='bold')
axes[3].tick_params(labelsize=7)

plt.suptitle('Channel-wise Correlation Matrices by Class', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. 뇌 영역 간 연결 강도 (5x5)

In [ ]:
# 19x19 → 5x5 영역 집계
region_names = list(BRAIN_REGIONS.keys())

def channel_corr_to_region_corr(corr_19x19):
    """19x19 채널 상관 행렬을 5x5 영역 상관 행렬로 변환"""
    n_regions = len(region_names)
    region_corr = np.zeros((n_regions, n_regions))
    
    for i, r1 in enumerate(region_names):
        for j, r2 in enumerate(region_names):
            ch_i = BRAIN_REGIONS[r1]
            ch_j = BRAIN_REGIONS[r2]
            
            if i == j:
                # 같은 영역 내: 대각선 제외 평균
                vals = []
                for ci in ch_i:
                    for cj in ch_j:
                        if ci != cj:
                            vals.append(corr_19x19[ci, cj])
                region_corr[i, j] = np.mean(vals) if vals else 1.0
            else:
                # 다른 영역 간: 모든 쌍 평균
                vals = [corr_19x19[ci, cj] for ci in ch_i for cj in ch_j]
                region_corr[i, j] = np.mean(vals)
    
    return region_corr

# Class별 5x5 영역 연결 행렬
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

region_corr_by_class = {}
for ax_idx, cls in enumerate(['Normal', 'MCI', 'Dementia']):
    region_corr_by_class[cls] = channel_corr_to_region_corr(avg_corr[cls])
    
    sns.heatmap(region_corr_by_class[cls], annot=True, fmt='.3f', cmap='YlOrRd',
                xticklabels=region_names, yticklabels=region_names,
                ax=axes[ax_idx], square=True, vmin=0, vmax=0.8)
    axes[ax_idx].set_title(f'{cls}', fontweight='bold')

# Difference
diff_region = region_corr_by_class['Normal'] - region_corr_by_class['Dementia']
sns.heatmap(diff_region, annot=True, fmt='.3f', cmap='RdBu_r',
            xticklabels=region_names, yticklabels=region_names,
            ax=axes[3], square=True, vmin=-0.15, vmax=0.15)
axes[3].set_title('Normal - Dementia', fontweight='bold')

plt.suptitle('Region-level Connectivity (5x5)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# 가장 큰 차이를 보이는 연결
print("\n=== Normal-Dementia 연결 강도 차이 (Top 5) ===")
diffs = []
for i in range(5):
    for j in range(i+1, 5):
        diffs.append({
            'connection': f'{region_names[i]}-{region_names[j]}',
            'Normal': region_corr_by_class['Normal'][i, j],
            'Dementia': region_corr_by_class['Dementia'][i, j],
            'diff': diff_region[i, j],
        })
df_diffs = pd.DataFrame(diffs).sort_values('diff', key=abs, ascending=False)
print(df_diffs.to_string(index=False))

## 3. Graph Metrics

간단한 그래프 메트릭(평균 연결 강도, clustering coefficient)을 class별로 비교합니다.

In [ ]:
# 녹음별 graph metrics 계산
def compute_graph_metrics(corr_matrix, threshold=0.3):
    """상관 행렬에서 간단한 그래프 메트릭 계산"""
    n = corr_matrix.shape[0]
    
    # 이진 인접 행렬 (threshold 이상)
    adj = (np.abs(corr_matrix) > threshold).astype(float)
    np.fill_diagonal(adj, 0)
    
    # 평균 연결 강도 (상삼각만)
    triu_idx = np.triu_indices(n, k=1)
    mean_connectivity = np.mean(np.abs(corr_matrix[triu_idx]))
    
    # Degree (연결 수)
    degrees = adj.sum(axis=1)
    mean_degree = degrees.mean()
    
    # Clustering coefficient (간단 버전)
    cc_list = []
    for i in range(n):
        neighbors = np.where(adj[i] > 0)[0]
        k = len(neighbors)
        if k < 2:
            cc_list.append(0)
            continue
        # 이웃 간 연결 수
        links = sum(adj[ni, nj] for ni in neighbors for nj in neighbors if ni < nj)
        max_links = k * (k - 1) / 2
        cc_list.append(links / max_links if max_links > 0 else 0)
    clustering_coeff = np.mean(cc_list)
    
    return {
        'mean_connectivity': mean_connectivity,
        'mean_degree': mean_degree,
        'clustering_coeff': clustering_coeff,
    }

# 전체 학습 데이터에서 메트릭 계산
graph_records = []
for cls in ['Normal', 'MCI', 'Dementia']:
    for corr_mat in class_corr_matrices[cls]:
        metrics = compute_graph_metrics(corr_mat, threshold=0.3)
        metrics['class_name'] = cls
        graph_records.append(metrics)

df_graph = pd.DataFrame(graph_records)

# 시각화
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
metrics_to_plot = ['mean_connectivity', 'clustering_coeff', 'mean_degree']
titles = ['Mean Connectivity', 'Clustering Coefficient', 'Mean Degree']

for i, (metric, title) in enumerate(zip(metrics_to_plot, titles)):
    sns.boxplot(data=df_graph, x='class_name', y=metric,
                order=['Normal', 'MCI', 'Dementia'], palette=colors, ax=axes[i])
    axes[i].set_title(title)

plt.tight_layout()
plt.show()

# 통계 검정
print("=== Graph Metrics 통계 ===")
for metric in metrics_to_plot:
    print(f"\n{metric}:")
    groups = []
    for cls in ['Normal', 'MCI', 'Dementia']:
        vals = df_graph[df_graph['class_name'] == cls][metric]
        print(f"  {cls}: mean={vals.mean():.4f}, std={vals.std():.4f}")
        groups.append(vals)
    
    stat, p = stats.kruskal(*groups)
    print(f"  Kruskal-Wallis: H={stat:.3f}, p={p:.2e}")

## 4. EDCC GCN 인접 행렬 설계 근거

위 분석 결과를 바탕으로, EDCC Stage 3 GCN에서 사용할 고정 인접 행렬의 설계 근거를 정리합니다.

**확인 사항**:
- [ ] 해부학적 근접 영역 간 연결이 강한지 → 고정 인접 행렬 설계 근거
- [ ] Dementia에서 특정 영역 간 연결이 약화되는지 → GCN이 포착해야 할 패턴
- [ ] MCI의 연결 패턴이 Normal과 Dementia 사이인지 → 점진적 변화 포착 가능성
- [ ] Graph metrics가 class별로 유의미하게 다른지 → GCN 필요성 뒷받침

**GCN 인접 행렬 후보**:
1. 해부학적 근접성 기반 (고정): Frontal-Central, Central-Parietal, Central-Temporal, Parietal-Temporal, Parietal-Occipital
2. 학습 데이터의 평균 상관관계 기반 (반고정): 위 분석에서 계산된 5x5 행렬
3. 완전 학습 가능 (학습): GCN 내에서 attention으로 edge weight 학습

**Phase 1 EDA 전체 완료.** Phase 2 (Baseline) 및 Phase 3 (EDCC 구현)으로 진행합니다.